In [ ]:
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib as mpl
import matplotlib.patheffects as mpe
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import rasterio as rio
import seaborn as sns
from scipy.stats import linregress
from tqdm import tqdm
import xarray as xr
from scipy.stats import mannwhitneyu, ttest_ind, wilcoxon

from project.utils.data import enrich, load_hillshade, sel_subset
from project.utils.metrics import compute_metrics
from project.utils.paths import get_path
from project.utils.plotting import ProjectCustomExtent, despine, cbar_halo

plt.rcParams["figure.dpi"] = 150

# Switzerland LV95 with extended limits to cover 'Hydrological Switzerland'.
lv95 = ProjectCustomExtent(2056, [2474273, 2842000, 1058000, 1325000])

In [ ]:
# Load predictions.
ds = xr.open_zarr(get_path('predictions_file')).sel(
    time=slice('1993', None))

# Load HRU polygons.
hrus = gpd.read_file(
    get_path('hru_file'), use_arrow=True).set_index('EZGNR')

metrics_max = ['kge', 'nse', 'nnse', 'r']
metrics_min = ['rmse', 'bias2', 'varerr', 'phaseerr', 'relbias', 'relscale', 'flv', 'fhv']

# Compute metrics and add them to dataset.
met = compute_metrics(ds.Q_obs_m3s, ds.Q_pred_m3s, metrics=metrics_min + metrics_max, dim='time')
ds = xr.merge([ds, met])

for m in metrics_min + metrics_max:
    name = f'{m}_impr'

    # Create the target variable once per metric.
    # Use the metric itself as a template (and make sure it's float so NaN works).
    if name not in ds:
        ds[name] = xr.full_like(ds[m], np.nan, dtype=float).load()

    for model in ds.model.values:
        # Pair model with the corresponding other model
        model_type, split_type = model.split('_')
        other_model_type = 'routed' if model_type == 'lumped' else 'lumped'
        other_model = f'{other_model_type}_{split_type}'
        if other_model not in ds.model.values:
            continue

        value_model = ds.sel(model=model)[m]
        value_other = ds.sel(model=other_model)[m]

        if m in metrics_min:
            value_model = np.abs(value_model)
            value_other = np.abs(value_other)

        if m in metrics_min:
            # Lower is better
            pec_impr = (value_other - value_model) / (value_other + 1e-6) * 100.0
        else:
            # Higher is better
            pec_impr = (value_model - value_other) / (1 - value_other) * 100.0

        # Write only the slice for this model
        ds[name].loc[dict(model=model)] = pec_impr

rivers = gpd.read_file(get_path('raw_data_root') / 'raw/rivernet', use_arrow=True)

rivers_subset = rivers.loc[rivers.CUM_LEN > 20000]

bg = load_hillshade(
    hillshade_dir=get_path('raw_data_root') / 'processed',
    dem_path=get_path('raw_data_root') / 'raw/merit.zarr',
    hrus=hrus,
    ds=ds,
    use_terrain=False,
    force_recompute=False)

In [ ]:
def plot_trend_map(
        da: xr.DataArray,
        title: str,
        label: str,
        cmap: str | None = None,
        q_min: float = 0.02,
        q_max: float = 0.98,
        use_rivers: bool = False,
        ax: mpl.axes.Axes | None = None) -> None:
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4), subplot_kw={'projection': lv95})

    mn, mx = da.quantile([q_min, q_max]).values

    bodr = cartopy.feature.NaturalEarthFeature(category='cultural', 
        name='admin_0_boundary_lines_land', scale='10m', facecolor='none')

    ax.set_global()

    # bg.plot.imshow(ax=ax, interpolation='bilinear', alpha=0.8, zorder=0)
    ax.add_feature(bodr, linestyle='--', edgecolor='k', alpha=1, zorder=10, lw=0.7)

    if mn >=0 or mx <= 0:
        if cmap is None:
            cmap = 'RdPu'
        norm = mpl.colors.Normalize(vmin=mn, vmax=mx)
        extend = 'both'
    else:
        if cmap is None:
            cmap = 'coolwarm'

        if np.abs(mn) > np.abs(mx):
            extend = 'min'
        else:
            extend = 'max'

        mn, mx = min(mn, -mx), max(mx, -mn)
        norm = mpl.colors.Normalize(vmin=mn, vmax=mx)

    if use_rivers:
        hrus_T = enrich(
                ds=da,
                hrus=hrus,
                variables='trend'
            )
        rivers_cut = gpd.overlay(
            rivers,
            hrus_T.reset_index()[['EZGNR', 'trend', 'geometry']],
            how='intersection'
        )
        rivers_Q = rivers_cut.explode(index_parts=False).reset_index(drop=True)

        rivers_Q = rivers_Q.sort_values('trend', ascending=True)

        rw = np.clip(np.log(rivers_Q.CUM_LEN + 1) / 15 - 0.1, 0.3, None)

        # rivers_Q.plot(
        #     color='k',
        #     # linewidth=rw + 0.8,
        #     lw=1.0,
        #     zorder=2,
        #     capstyle='round',
        #     legend=False,
        #     ax=ax)

        rivers_Q.plot(
            column='trend',
            linewidth=rw + 0.2,
            lw=0.5,
            norm=norm,
            zorder=2,
            capstyle='round',
            fc='none',
            alpha=0.8,
            cmap=cmap,
            legend=False,
            ax=ax)

        lakes = hrus_T.loc[ds.hru.where(ds.is_lake.load() & (ds.area.load() > 2), drop=True)]

        lakes.plot(
            ec='k',
            fc='w',
            lw=0.2,
            alpha=1,
            legend=False,
            zorder=30,
            ax=ax
        )
        lakes.plot(
            column='trend',
            ec='w',
            lw=0.0,
            cmap=cmap,
            alpha=0.8,
            legend=False,
            zorder=30,
            ax=ax
        )

    else:
        enrich(
            ds=da,
            hrus=hrus,
            variables='trend'
        ).plot(
            column='trend',
            cmap=cmap,
            norm=norm,
            legend=False,
            ec='k',
            lw=0.1,
            alpha=0.6,
            vmin=mn,
            vmax=mx,
            ax=ax,
        )

    axins = ax.inset_axes((0.05, 0.90, 0.25, 0.02))
    mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = plt.colorbar(
        mappable,
        cax=axins,
        orientation='horizontal',   # or 'vertical'
        extend=extend,
        label=label,
    )

    cbar_halo(cbar=cbar, linewidth=0, color='w', labelsize=6, ticksize=6)

    # cbar.ax.xaxis.label.set_size(6)
    # cbar.ax.tick_params(labelsize=6)

    ax.set_title(title, fontsize=8)


In [ ]:
def compute_trends(ds_stats):
    ds_annual = ds_stats.assign_coords(time=ds_stats.time.dt.year)
    ds_trend = ds_annual.polyfit(dim='time', deg=1, skipna=True).sel(degree=1)
    ds_trend = ds_trend.rename({var: var.replace('_polyfit_coefficients', '') for var in ds_trend.data_vars})
    ds_trend = ds_trend.where(ds_trend != 0) * 10
    ds_trend_perc = (ds_trend / ds_annual.mean('time')).compute() * 100

    return ds_trend, ds_trend_perc

In [ ]:
ds_sel = ds.sel(model='routed_random')[['Q_pred_mmd', 'q_pred_mmd', 'prec', 'tair', 'area'] + [var for var in ds.data_vars if ds[var].dims == ('hru',)]].load()

In [ ]:
ds_annual = xr.Dataset({
    'Q_pred_mmd': ds_sel.Q_pred_mmd.resample(time='1YE').mean(skipna=True),
    'q_pred_mmd': ds_sel.q_pred_mmd.resample(time='1YE').mean(skipna=True),
    'prec': ds_sel.prec.resample(time='1YE').sum(skipna=True),
    'tair': ds_sel.tair.resample(time='1YE').mean(skipna=True),
}).compute()

ds_annual_max = xr.Dataset({
    'Q_pred_mmd': ds_sel.Q_pred_mmd.resample(time='1YE').max(skipna=True),
    'q_pred_mmd': ds_sel.q_pred_mmd.resample(time='1YE').max(skipna=True),
    'prec': ds_sel.prec.rolling(time=3, center=True).sum().resample(time='1YE').max(skipna=True),
    'tair': ds_sel.tair.rolling(time=3, center=True).mean().resample(time='1YE').max(skipna=True),
}).compute()

ds_annual_low = xr.Dataset({
    'Q_pred_mmd': -(ds_sel.Q_pred_mmd - ds_sel.Q_pred_mmd.quantile(0.02, dim='time')).clip(max=0).resample(time='1YE').sum(),
    'q_pred_mmd': -(ds_sel.q_pred_mmd - ds_sel.q_pred_mmd.quantile(0.02, dim='time')).clip(max=0).resample(time='1YE').sum(),
    'prec': ds_sel.prec.rolling(time=14, center=True).sum().resample(time='1YE').min(skipna=True),
    'tair': ds_sel.tair.rolling(time=14, center=True).mean().resample(time='1YE').max(skipna=True),
}).compute()

ds_trend, ds_trend_perc = compute_trends(ds_annual)
ds_trend_max, ds_trend_max_perc = compute_trends(ds_annual_max)
ds_trend_low, ds_trend_low_perc = compute_trends(ds_annual_low)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 5.5), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.0})

plot_trend_map(
    ds_trend.prec, 'Precipitation annual sum', 'mmy$^{-1}$ per decade', ax=axes[0, 0], cmap='bwr_r')
plot_trend_map(
    ds_trend.tair, 'Temperature annual mean', '°C per decade', ax=axes[0, 1], cmap='Oranges')
plot_trend_map(
    ds_trend_perc.q_pred_mmd, 'Runoff annual mean', '% per decade', ax=axes[1, 0], cmap='BrBG')
plot_trend_map(
    ds_trend_perc.Q_pred_mmd, 'Discharge annual mean', '% per decade', ax=axes[1, 1], cmap='BrBG', use_rivers=True)

fig.savefig('./figures/trends_mean.png', dpi=300, bbox_inches='tight')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 5.5), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.0})

plot_trend_map(
    ds_trend_max.prec, '3-day precipitation sum annual max', 'mm per decade', ax=axes[0, 0], cmap='bwr_r')
plot_trend_map(
    ds_trend_max.tair, '3-day temperaturen mean annual max', '°C per decade', ax=axes[0, 1], cmap='Oranges')
plot_trend_map(
    ds_trend_max_perc.q_pred_mmd, 'Runoff annual max', '% per decade', ax=axes[1, 0], cmap='BrBG')
plot_trend_map(
    ds_trend_max_perc.Q_pred_mmd, 'Dischargen annual max', '% per decade', ax=axes[1, 1], cmap='BrBG', use_rivers=True)

fig.savefig('./figures/trends_max.png', dpi=300, bbox_inches='tight')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 5.5), subplot_kw={'projection': lv95}, gridspec_kw={'wspace': 0.0})

plot_trend_map(
    ds_trend_low.prec, '2-week precipitation sum annual min', 'mm per decade', ax=axes[0, 0], cmap='bwr_r')
plot_trend_map(
    ds_trend_low.tair, '2-week temperaturen mean annual max', '°C per decade', ax=axes[0, 1], cmap='Oranges')
plot_trend_map(
    ds_trend_low_perc.q_pred_mmd, 'Runoff low flow (<2%) annual deficit', '% per decade', ax=axes[1, 0], cmap='BrBG_r')
plot_trend_map(
    ds_trend_low_perc.Q_pred_mmd, 'Dischargen low flow (<2%) annual deficit', '% per decade', ax=axes[1, 1], cmap='BrBG_r', use_rivers=True)

fig.savefig('./figures/trends_low.png', dpi=300, bbox_inches='tight')


In [ ]:
plot_trend_map(
    ds_Q_trend_max,
    ['Q_pred_mmd', 'q_pred_mmd'],
    ['Streamflow maximum trend', 'Runoff maximum trend (PREVAH)'],
    units='mm/d/year',
    cmap='BrBG'
)

In [ ]:
ds_Q_trend_max.prec.plot.hist(bins=50)

In [ ]:
plot_trend_map(
    ds_Q_trend_max,
    ['prec'],
    ['Precipitation'],
    units='mm/d/year',
    cmap='BrBG',
    q_min=0.05,
    q_max=0.95
)

In [ ]:
def hexbin(x, y, ax, xlabel='x', ylabel='y', show_fit = True):

    hexbin_kwargs = dict(
        lw=1, bins='log', mincnt=1, gridsize=30, ec='w', cmap='Blues'
    )

    hb = plt.hexbin(x, y, **hexbin_kwargs)  # no extent
    verts = hb.get_offsets()  # returns centers of all hexagons
    hb.remove()
    x_coords, y_coords = verts[:,0], verts[:,1]
    x_min = x_coords.min()
    x_max = x_coords.max()
    y_min = y_coords.min()
    y_max = y_coords.max()

    ax.hexbin(x, y, extent=(x_min, x_max, y_min, y_max), **hexbin_kwargs)
    # ax.plot([x_min, mx], [mn, mx], '--', color='k', lw=1)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    if show_fit:
        # Linear regression
        slope, intercept, r_value, p_value, std_err = linregress(x, y)

        # Create regression line
        x_fit = np.linspace(x_min, x_max, 2)
        y_fit = slope * x_fit + intercept
        ax.plot(x_fit, y_fit, color='w', lw=2.5)
        ax.plot(x_fit, y_fit, color='tab:blue', lw=1.5, label='Linear fit')

        # Equation text (inset)
        eq_text = f"y = {slope:.2f}x + {intercept:.2f}\nR² = {r_value**2:.2f}"
        ax.text(0.05, 0.95, eq_text,
                transform=ax.transAxes,
                fontsize=9, va='top', ha='left', color='tab:blue', 
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

In [ ]:
levels = [
    0, 500, 1000, 1500, 2000
]

fig, axes = plt.subplots(2, len(levels)- 1, figsize=(4 * (len(levels) - 1), 8))

for i in range(len(levels) - 1):
    axc = axes[:, i]

    lower = levels[i]
    upper = levels[i + 1]

    subset = ds_Q_trend_perc.where(
        ((ds.height_mean >= lower) & (ds.height_mean < upper)).compute(), drop=True
    )

    ax = axc[0]

    ax.set_title(f'Elevation: {lower} - {upper} m a.s.l.')

    hexbin(
        subset.q_pred_m3s,
        subset.prec,
        ax=ax,
        xlabel='',
        ylabel='Precipitation trend (%/year)' if i == 0 else '',
        show_fit=True
    )

    ax = axc[1]

    hexbin(
        subset.q_pred_m3s,
        subset.tair,
        ax=ax,
        xlabel='Runoff trend (%/year)',
        ylabel='Precipitation trend (%/year)' if i == 0 else '',
        show_fit=True
    )

In [ ]:
plot_trend_map(
    ds_Q_trend_perc,
    ['Q_pred_m3s', 'q_pred_m3s'],
    ['Streamflow trend', 'Runoff trend'],
    units='%/year',
    cmap='BrBG')